Zadanie zaczynamy od sparsowania plików z danymi oraz załadowania ich do  Dataframeów z biblioteki Pandas. 
Usuwamy pliki **agency.csv** oraz **feed_info.csv**, jako że nie zawierają potrzebnych informacji

In [79]:
import os
import pandas as pd 
from pathlib import Path
data_path = os.path.join(os.getcwd(), "google_transit")
print(data_path)
entries = os.listdir(data_path)
entries =  [x for x in entries if x not in ["agency.csv", "feed_info.csv"]]
dfs = {}
for entry in entries:
    dfs[entry] = (pd.read_csv(os.path.join(data_path ,entry)))

c:\Users\mikka\Desktop\SEM6\SI\Kurs-Sztuczna-inteligencja-\Lista1\google_transit


Zdefiniujmy klasy którymi będziemy przedstawiać nasz graf: 
1. **Edge** - klasa krawędzi
2. **Node** - klasa wierzchołka
3. **Transit Graph** - klasa grafu, agreguje klasy **Node** oraz **Edge**

In [80]:
from dataclasses import dataclass
import datetime


@dataclass 
class BaseEdge:
    target_stop_id: str
    is_transfer: bool

@dataclass
class TransferEdge(BaseEdge): 
    transfer_time: int = 60

@dataclass 
class RegularEdge(BaseEdge):
    target_stop_id: str
    departure_time: int # sekundy od północy
    arrival_time: int 
    route_name: str 
    service_id: str
    is_transfer = False

@dataclass
class Node: 
    stop_id: str
    stop_name: str
    stop_lat: float
    stop_lon: float 
    type: int
    parent_station: str 

#klasa dla odtwarzania tras
@dataclass
class RouteObject:
    orgin_node_id: str
    edge_used: BaseEdge
    is_transfer: bool


class TransitCalendar:
    def __init__(self):
        # service_id -> { 'start': 'YYYYMMDD', 'end': 'YYYYMMDD', 'days': [mon, tue, wed, thu, fri, sat, sun] }
        self.regular_schedules = {}
        # service_id -> { 'added': set(), 'removed': set() }
        self.exceptions = {}
    def set_schedules(self, schedules: dict): 
        self.regular_schedules = schedules
    def set_exceptions(self, exceptions: dict): 
        self.exceptions = exceptions

    def is_active(self, service_id: str, date_str: str) -> bool:
        """
        Sprawdza, czy pociąg faktycznie jedzie w podanym dniu.
        Format daty to YYYYMMDD, np. '20260308'.
        """
        if service_id in self.exceptions:
            if date_str in self.exceptions[service_id]['removed']:
                return False
            if date_str in self.exceptions[service_id]['added']:
                return True
                
        if service_id not in self.regular_schedules:
            return False
            
        schedule = self.regular_schedules[service_id]
        
        if not (schedule['start'] <= date_str <= schedule['end']):
            return False
            
        year = int(date_str[:4])
        month = int(date_str[4:6])
        day = int(date_str[6:8])
        weekday = datetime.date(year, month, day).weekday()        
        return schedule['days'][weekday] == 1

class TransitGraph():
    def __init__(self):
        self.nodes : dict[str ,Node] = {}
        self.adjacent : dict[str, list[BaseEdge]] = {} # klucz id wierzchołka, wartość lista krawędzie skierowanych od tego wierzchołka
        self.calendar: TransitCalendar = {}
    def add_nodes(self, nodes: list[Node]): 
        for entry in nodes: 
            self.nodes[entry.stop_id]=entry
    def add_edge(self, source_node_id: str,edge: BaseEdge):
        curr = self.adjacent.get(source_node_id, [])
        curr.append(edge)
        self.adjacent[source_node_id] = curr
    def add_edges(self, edge_dict: dict[str, BaseEdge]):
        for key, value in edge_dict.items():
            curr = self.adjacent.get(key, [])
            curr += value  
            self.adjacent[key] = curr 
    def set_calendar(self, calendar: TransitCalendar): 
        self.calendar= calendar
    def get_valid_neighbours(self, source_node_id: str, current_time: int, current_date: str) -> list:
        valid_moves = [] 
        neighbours = self.adjacent.get(source_node_id, []) 
        
        for edge in neighbours:
            if isinstance(edge, TransferEdge):
                continue
                
            if edge.departure_time >= current_time and self.calendar.is_active(edge.service_id, current_date):
                valid_moves.append((edge, edge.arrival_time, False))
                
        for transfer_edge in neighbours: 
            if not isinstance(transfer_edge, TransferEdge):
                continue
                
            time_after_transfer = current_time + transfer_edge.transfer_time
            target_platform = transfer_edge.target_stop_id
            
            next_neighbours = self.adjacent.get(target_platform, [])
            for next_edge in next_neighbours: 
                if isinstance(next_edge, TransferEdge):
                    continue
                    
                if next_edge.departure_time >= time_after_transfer and self.calendar.is_active(next_edge.service_id, current_date):
                    valid_moves.append((next_edge, next_edge.arrival_time, True))
                    
        return valid_moves
    
    def check_content(self): 
        print("Nodes count", len(self.nodes.keys()))
        print("Edges count", sum([len(x) for x in self.adjacent.values()]))

Przetwarzamy utworzone wcześniej Dataframy, do postaci zdefiniowanych klas


In [81]:
df_stops = dfs["stops.csv"]
df_routes = dfs["routes.csv"]
df_trips = dfs["trips.csv"]
df_stop_times = dfs["stop_times.csv"]
df_calendar  = dfs["calendar.csv"]
df_calendar_dates = dfs["calendar_dates.csv"]


def time_to_seconds(time_str: str) -> int:
    if pd.isna(time_str):
        return 0
    h, m, s = map(int, str(time_str).split(':'))
    return h * 3600 + m * 60 + s

def load_stops(df: pd.DataFrame):
    res = []
    grouped = {}
    
    for row in df.itertuples():
        # 1. Pobieramy ID i usuwamy potencjalne .0 na końcu
        current_stop_id = str(row.stop_id)
        if current_stop_id.endswith('.0'):
            current_stop_id = current_stop_id[:-2]
            
        # 2. To samo dla parent_station
        parent_station_id = ""
        if pd.notna(row.parent_station):
            parent_station_id = str(row.parent_station)
            if parent_station_id.endswith('.0'):
                parent_station_id = parent_station_id[:-2]

        node = Node(
            stop_id = current_stop_id, 
            stop_name = str(row.stop_name), 
            stop_lat = float(row.stop_lat), 
            stop_lon = float(row.stop_lon), 
            type = int(row.location_type) if pd.notna(row.location_type) else 0, 
            parent_station = parent_station_id,  
        )
        res.append(node)
        
        # 3. Teraz klucze w słowniku wreszcie się pokryją!
        group_key = parent_station_id if parent_station_id != "" else current_stop_id
        
        cr_list = grouped.get(group_key, [])
        cr_list.append(node)
        grouped[group_key] = cr_list
        
    print(f"Created {len(res)} nodes")
    return (res, grouped)



def create_transfer_edges(grouped: dict[str, list], const_transfer_time: int ):
    transfer_edges = {}
    transfer_count = 0
    for parent_id, nodes in grouped.items():
        if len(nodes) < 2:
            continue
            
        for source_node in nodes:
            cr_transfer =[]
            for target_node in nodes:
                if source_node.stop_id != target_node.stop_id:
                    
                    
                    transfer_time_sec =const_transfer_time
                    
                    transfer_edge = TransferEdge(
                        target_stop_id=target_node.stop_id,
                        transfer_time=transfer_time_sec,
                        is_transfer=True
                    )
                    
                    cr_transfer.append(transfer_edge)
            transfer_edges[source_node.stop_id] = cr_transfer
            transfer_count+=len(cr_transfer)
                    
    print(f"Created {transfer_count} transfer edges")
    return transfer_edges
                    

In [82]:
def load_edges(df_routes: pd.DataFrame, df_trips: pd.DataFrame, df_stop_times: pd.DataFrame, graph: 'TransitGraph'):
    routes = {}
    trips = {}
    count = 1
    
    for row in df_routes.itertuples():
        if pd.isna(row.route_short_name) or str(row.route_short_name).strip() == "":
            routes[row.route_id] = str(row.route_long_name)
        else:
            routes[row.route_id] = str(row.route_short_name)
            
    for row in df_trips.itertuples(): 
        trips[row.trip_id] = {
            "route_id": row.route_id, 
            "service_id": row.service_id
        }
        
    df_stop_times = df_stop_times.sort_values(["trip_id", "stop_sequence"])
    
    prev = None 
    for row in df_stop_times.itertuples():
        if prev and prev.trip_id == row.trip_id:
            trip_info = trips[row.trip_id]
            route_name = routes[trip_info["route_id"]]
            
            edge = RegularEdge(
                target_stop_id=str(row.stop_id),
                departure_time=time_to_seconds(prev.departure_time), 
                arrival_time=time_to_seconds(row.arrival_time),     
                route_name=route_name,
                service_id=str(trip_info["service_id"]),
                is_transfer=False
            )
            
            graph.add_edge(str(prev.stop_id), edge)
            count+=1

        prev = row
    print(f"Created {count} edges")

In [83]:
def load_calendar(df_calendar: pd.DataFrame):
    regular_schedules = {}
    for row in df_calendar.itertuples():
        regular_schedules[str(row.service_id)] = {
            'start': str(row.start_date),
            'end': str(row.end_date),
            'days': [
                row.monday, row.tuesday, row.wednesday, 
                row.thursday, row.friday, row.saturday, row.sunday
            ]
        }
    print(f"Created {len(regular_schedules)} schedules")
    return regular_schedules

def load_calendar_dates(df_calendar_dates: pd.DataFrame):
    exceptions = {}
    for row in df_calendar_dates.itertuples():
        sid = str(row.service_id)
        date_str = str(row.date)
        ex_type = int(row.exception_type)
        
        if sid not in exceptions:
            exceptions[sid] = {'added': set(), 'removed': set()}
            
        if ex_type == 1:
            exceptions[sid]['added'].add(date_str)
        elif ex_type == 2:
            exceptions[sid]['removed'].add(date_str)
    print(f"Created {len(exceptions)} exceptions")
    return exceptions 


In [84]:
graph = TransitGraph()
nodes, grouped_nodes = load_stops(df_stops)
graph.add_nodes(nodes)

transfer_edges = create_transfer_edges(grouped_nodes, 60)
graph.add_edges(transfer_edges)
load_edges(df_routes, df_trips, df_stop_times, graph)


calendar = TransitCalendar()
calendar.set_exceptions(load_calendar_dates(df_calendar_dates))
calendar.set_schedules(load_calendar(df_calendar))
graph.set_calendar(calendar)

graph.check_content()

Created 1108 nodes
Created 2362 transfer edges
Created 45486 edges
Created 1123 exceptions
Created 3263 schedules
Nodes count 1108
Edges count 47847


Helpery i funkcje kosztu dla algorytmów

In [85]:

#odtworzenie ścieżki algorytmu 
def reconstruct_path(came_from: dict, current_id: str): 
    path = []
    while current_id in came_from:
        entry = came_from[current_id]
        routeob = RouteObject(entry['prev_node'], entry['edge_used'], entry['is_transfer'])
        path.append(routeob)
        current_id = routeob.orgin_node_id
    path.reverse()
    return path

import math 
#funckja do liczenia dystansu
def calc_distance(st_lat, st_lon, ed_lat, ed_lon):
    R = 6371000.0 
    
    lat1 = math.radians(st_lat)
    lon1 = math.radians(st_lon)
    lat2 = math.radians(ed_lat)
    lon2 = math.radians(ed_lon)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    distance = R * c
    
    return distance


#funckje kosztu dla kryterium t
def g_cost_time(current_g, is_transfer, arrival_time, is_start):
    return arrival_time

def h_cost_time(node, target_node):
    dist = calc_distance(node.stop_lat, node.stop_lon, target_node.stop_lat, target_node.stop_lon)
    v_ms = 44.0 
    return dist / v_ms

#funkcje kosztu dla kryterium p 
def g_cost_transfers(current_g, is_transfer, arrival_time, is_start):
    time_penalty = arrival_time / 10000000.0
    
    if is_start: 
        return time_penalty
        
    base_transfers = int(current_g)
    
    if is_transfer:
        base_transfers += 1
        
    return base_transfers + time_penalty

def h_cost_transfers(node, target_node):
    # Zwracamy 0 (Algorytm Dijkstry dla przesiadek)
    return 0



Implementacja algorytmu A*

In [ ]:
import heapq
def A_star(start_id: str, end_id: str, start_time: int, start_date: str, graph: TransitGraph, h_cost_func, g_cost_func):
    open_set = []
    
    # Inicjalizacja początkowego kosztu 'g' za pomocą Twojej zewnętrznej funkcji
    start_g = g_cost_func(current_g=0, is_transfer=False, arrival_time=start_time, is_start=True)
    
    g_scores = {start_id: start_g}
    came_from = {}
    
    start_node = graph.nodes[start_id]
    end_node = graph.nodes[end_id]
    h_start = h_cost_func(start_node, end_node) 
    
    #(f_cost, g_cost, czas, id_węzła)
    heapq.heappush(open_set, (start_g + h_start, start_g, start_time, start_id))
    
    closed_set = set()

    while len(open_set) > 0:
        curr_f, curr_g, curr_time, curr_id = heapq.heappop(open_set)

        curr_node = graph.nodes[curr_id]
        is_goal = (curr_id == end_id)
        if not is_goal and curr_node.parent_station != "":
            if curr_node.parent_station == end_id:
                is_goal = True
            # Dodatkowo, jeśli użytkownik wpisał ID peronu, a my jesteśmy na innym peronie tej samej stacji
            elif curr_node.parent_station == end_node.parent_station:
                is_goal = True
                
        if is_goal:
            print("Znaleziono rozwiązanie!")
            return reconstruct_path(came_from, curr_id)
            
        if curr_id in closed_set:
            continue
            
        closed_set.add(curr_id)
        
        # Szukamy sąsiadów używając OBECNEGO CZASU (curr_time), a nie kosztu!
        neighbours = graph.get_valid_neighbours(curr_id, curr_time, start_date)
        
        for edge, arrival_time, is_transfer in neighbours:
            next_id = edge.target_stop_id
            
            if next_id in closed_set:
                continue
                
            # NOWOŚĆ: Wyliczamy nowy koszt G uniwersalną funkcją
            tentative_g = g_cost_func(
                current_g=curr_g, 
                is_transfer=is_transfer, 
                arrival_time=arrival_time, 
                is_start=False
            )
            
            if next_id not in g_scores or tentative_g < g_scores[next_id]:
                came_from[next_id] = {
                    'prev_node': curr_id,
                    'edge_used': edge,
                    'is_transfer': is_transfer
                }
                
                g_scores[next_id] = tentative_g
                
                # Używamy przekazanej funkcji do heurystyki
                next_node_obj = graph.nodes[next_id]
                h_cost = h_cost_func(next_node_obj, end_node)
                
                f_cost = tentative_g + h_cost
                
                # Wrzucamy do kolejki nową krotkę (pamiętając o arrival_time jako nowym curr_time!)
                heapq.heappush(open_set, (f_cost, tentative_g, arrival_time, next_id))
                
    return None





Implementacja Dijkstry

Helpery do wyświetlania

In [87]:
def format_time(seconds: int) -> str:
    """Zamienia sekundy od północy na format HH:MM:SS"""
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h >= 24:
        h -= 24
    return f"{h:02d}:{m:02d}:{s:02d}"

def get_stop_name(stops_df: pd.DataFrame, stop_id: str) -> str:
    """Bezpiecznie wyciąga nazwę przystanku na podstawie jego ID z DataFrame'a"""
    matching_stops = stops_df[stops_df['stop_id'].astype(str) == str(stop_id)]
    
    if not matching_stops.empty:
        return str(matching_stops['stop_name'].values[0])
    return f"Nieznany Przystanek (ID: {stop_id})"

def res_printing(path: list, stops_df: pd.DataFrame):
    if not path:
        print("Trasa jest pusta! Algorytm nie znalazł połączenia.")
        return

    print("=" * 60)
    print("SZCZEGÓŁY TWOJEJ PODRÓŻY")
    print("=" * 60)

    transfer_count = 0
    
    start_time = path[0].edge_used.departure_time
    end_time = path[-1].edge_used.arrival_time

    for i, entry in enumerate(path):
        origin_id = entry.orgin_node_id 
        target_id = entry.edge_used.target_stop_id
        
        origin_name = get_stop_name(stops_df, origin_id)
        target_name = get_stop_name(stops_df, target_id)
        
        dep_time = format_time(entry.edge_used.departure_time)
        arr_time = format_time(entry.edge_used.arrival_time)
        route = entry.edge_used.route_name
        
        if entry.is_transfer:
            transfer_count += 1
            print(f"[ZMIANA PERONU / PRZESIADKA na stacji {origin_name}]")
            
        print(f"{dep_time} | Odjazd:   {origin_name}")
        print(f"   |          (Linia: {route})")
        print(f"{arr_time} | Przyjazd: {target_name}")
        
        if i < len(path) - 1:
            print("-" * 40)


    total_time_minutes = (end_time - start_time) // 60

    print("=" * 60)
    print("PODSUMOWANIE")
    print("=" * 60)
    print(f"Trasa:             {get_stop_name(stops_df, path[0].orgin_node_id)} ➔ {get_stop_name(stops_df, path[-1].edge_used.target_stop_id)}")
    print(f"Godzina wyjazdu:   {format_time(start_time)}")
    print(f"Godzina przyjazdu: {format_time(end_time)}")
    print(f"Całkowity czas:    {total_time_minutes} minut")
    print(f"Liczba przesiadek: {transfer_count}")
    print("=" * 60)

Interfejsy dla zadań

In [88]:


def zad1(station_A: str, station_B: str, criteria: str, start_time: str, start_date: str, algorithm: str):
    print(f"\n--- TEST: {station_A} -> {station_B} o {format_time(start_time)} ---")
    
    # Sprawdźmy, czy te węzły w ogóle istnieją w grafie!
    if station_A not in graph.nodes:
        print(f"BŁĄD: Brak stacji startowej {station_A} w grafie!")
        return []
    if station_B not in graph.nodes:
        print(f"BŁĄD: Brak stacji docelowej {station_B} w grafie!")
        return []
        
    # Sprawdźmy, czy stacja startowa ma w ogóle jakieś krawędzie!
    neighbours = graph.get_valid_neighbours(station_A, start_time, start_date)
    print(f"Liczba fizycznie dostępnych odjazdów ze stacji startowej w tym dniu: {len(neighbours)}")
    match algorithm.lower(): 
        case 'a*':
            chosen = A_star
        case 'dijkstra': 
            # chosen = dijkstra
            return  
        case _: 
            print("Wybrano nieprawidłowy algorytm.")
            return

    match criteria.lower(): 
        case 't': 
            h = h_cost_time
            g = g_cost_time
        case 'p': 
            h = h_cost_transfers
            g = g_cost_transfers
        case _:
            print("Wybrano nieprawidłowe kryterium.")
            return 
            
    print(f"--- Szukam trasy z {station_A} do {station_B} ---")
    wynik = chosen(station_A, station_B, start_time,start_date, graph,h, g)
    return wynik




# res = zad1('1413380', '1474662', 't', 3660*9 , '20260525', 'a*')
data = '20260525'
time = 3600*9
tests = [('1413334', '1413380', 't', time, data, 'a*'), 
         ('1413334', '1413066', 't', time, data, 'a*'), 
         ('1413066', '1474861', 't', time, data, 'a*')]



time2 =time
data2 = data
tests_two_metrics= [('1413064', '2246783', time2, data2, 'a*'),
                    ('1413083', '1413104', time2, data2, 'a*'),
                    ('1413137', '1474618', time2, data2, 'a*')]

# res_printing(res, df_stops)


# for entry in tests:
#     res = zad1(entry[0],entry[1], entry[2], entry[3], entry[4], entry[5])
#     res_printing(res, df_stops)

for i, entry in enumerate(tests_two_metrics, 0):
    print(f"TEST {i+1} - czas")
    res = zad1(entry[0],entry[1], 't',entry[2], entry[3], entry[4])
    res_printing(res, df_stops)
    print(f"TEST {i+1}- przesiadki")
    res = zad1(entry[0],entry[1], 'p',entry[2], entry[3], entry[4])
    res_printing(res, df_stops)




TEST 1 - czas

--- TEST: 1413064 -> 2246783 o 09:00:00 ---
Liczba fizycznie dostępnych odjazdów ze stacji startowej w tym dniu: 0
--- Szukam trasy z 1413064 do 2246783 ---
Trasa jest pusta! Algorytm nie znalazł połączenia.
TEST 1- przesiadki

--- TEST: 1413064 -> 2246783 o 09:00:00 ---
Liczba fizycznie dostępnych odjazdów ze stacji startowej w tym dniu: 0
--- Szukam trasy z 1413064 do 2246783 ---
Trasa jest pusta! Algorytm nie znalazł połączenia.
TEST 2 - czas

--- TEST: 1413083 -> 1413104 o 09:00:00 ---
Liczba fizycznie dostępnych odjazdów ze stacji startowej w tym dniu: 20
--- Szukam trasy z 1413083 do 1413104 ---
Znaleziono rozwiązanie!
SZCZEGÓŁY TWOJEJ PODRÓŻY
[ZMIANA PERONU / PRZESIADKA na stacji Boguszów-Gorce Wschód]
10:11:00 | Odjazd:   Boguszów-Gorce Wschód
   |          (Linia: D60)
10:17:00 | Przyjazd: Wałbrzych Główny
----------------------------------------
10:17:00 | Odjazd:   Wałbrzych Główny
   |          (Linia: D60)
10:22:00 | Przyjazd: Wałbrzych Fabryczny
-----------